# Pelajaran 11 - Protokol Ejen-ke-Ejen (A2A)


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apakah Protokol A2A?

Protokol **Agen-ke-Agen (A2A)** adalah satu standard terbuka yang membolehkan agen AI berkomunikasi,
menemui antara satu sama lain, dan bekerjasama — walaupun mereka dibina pada rangka kerja yang berbeza atau dihoskan
oleh perkhidmatan yang berbeza.

Key concepts:

- **Penemuan** – Agen menerbitkan sebuah *Kad Agen* yang menerangkan kebolehan mereka, menjadikannya
  mudah bagi agen lain (atau pengorkestrasi) untuk mencari pakar yang tepat bagi sesuatu tugas.
- **Penghantaran Mesej** – Agen bertukar mesej berstruktur melalui protokol biasa, jadi satu
  permintaan daripada satu agen boleh difahami dan dipenuhi oleh agen lain tanpa mengira
  pelaksanaan dalaman mereka.
- **Kitaran Hayat Tugas** – A2A mentakrifkan keadaan seperti *dihantar*, *sedang diproses*, *selesai*, dan
  *gagal*, memberikan pengorkestrasi keterlihatan penuh tentang bagaimana tugasan yang diwakilkan sedang berjalan.

Dalam pelajaran ini kami mensimulasikan kerjasama gaya A2A dengan menghubungkan tiga agen pelancongan khusus
ke dalam aliran kerja di mana setiap agen menyumbangkan kepakaran mereka dan menyerahkan hasil kepada yang seterusnya.


## Mencipta Ejen Pelancongan Khusus


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kolaborasi Multi-Ejen melalui Aliran Kerja

Kami menyambungkan ketiga-tiga ejen ke dalam aliran kerja berurutan yang mencerminkan pemindahan mesej A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata wang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambah cadangan aktiviti.
3. **TravelManagerAgent** mensintesis kedua-dua input menjadi ringkasan perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Persekitaran Pengeluaran

Dalam persekitaran pengeluaran, protokol A2A membuka senario rentas-perkhidmatan yang berkuasa:

| Capability | Description |
|---|---|
| **Cross-framework interop** | Ejen yang dibina dengan satu rangka kerja boleh mendelegasikan tugas kepada ejen yang dibina dengan mana-mana rangka kerja lain yang mematuhi A2A, membolehkan interoperabiliti sebenar merentas organisasi. |
| **Service boundaries** | Ejen boleh berada dalam mikroperkhidmatan yang berasingan, rantau awan yang berbeza, atau bahkan organisasi yang berbeza sambil terus bekerjasama dengan lancar. |
| **Dynamic discovery** | Pengorkestrasi boleh membuat pertanyaan pada daftar Kad Ejen semasa pelaksanaan untuk mencari pakar yang paling sesuai bagi sub-tugas tertentu. |
| **Streaming & push notifications** | A2A menyokong Server-Sent Events (SSE) untuk kemas kini kemajuan masa nyata dan pemberitahuan push untuk tugas yang berjalan lama. |

Aliran kerja yang kami bina di atas adalah versi ringkas, dalam-proses bagi corak ini. Dalam penerapan sebenar setiap ejen akan mendedahkan titik akhir HTTP, menerbitkan Kad Ejen, dan berkomunikasi melalui protokol A2A JSON-RPC.


## Ringkasan

Dalam pelajaran ini anda telah belajar:

1. **Apa itu protokol A2A** — piawaian terbuka untuk penemuan agen-ke-agen, penghantaran mesej, dan pengurusan tugas.
2. **Cara mencipta agen khusus** — sebuah agen Pertukaran Mata Wang, sebuah agen Perancang Aktiviti, dan sebuah pengorkestrasi Pengurus Perjalanan.
3. **Cara menyambungkan agen ke dalam aliran kerja** — menggunakan `WorkflowBuilder` untuk memodelkan penghantaran mesej berurutan antara agen.
4. **Bagaimana A2A berfungsi dalam persekitaran produksi** — membolehkan kerjasama merentas rangka kerja dan merentas perkhidmatan dengan penemuan dinamik dan kemas kini penstriman.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha mencapai ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi ralat atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber rujukan utama. Untuk maklumat penting, disarankan mendapatkan terjemahan profesional oleh penterjemah manusia. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsiran yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
